In [2]:
# ID

prenom = "Chloé"
nom = "Makoundou"
student_id = "82506363"
formation = "M1 IBD"

print(f"Prénom : {prenom}\nNom : {nom}\nID étudiant : {student_id}\nFormation : {formation}")

Prénom : Chloé
Nom : Makoundou
ID étudiant : 82506363
Formation : M1 IBD


# **TP 2 — Préparation des données avec Python**

## Objectif
Nettoyer le dataset `catnat_dirty.csv` et produire un fichier propre `catnat_clean.csv` prêt à être analysé dans Tableau.

### Problèmes à résoudre
Le dataset contient volontairement :

- ~150 doublons
- Valeurs manquantes supplémentaires
- Incohérences de casse (Asia, ASIA, asia...)
- Espaces parasites
- Variantes d'orthographe (USA, US, United States...)
- `Start Year` en format texte avec erreurs ("2020 AD", "Year 2020")
- Outliers aberrants (décès négatifs, magnitude à 999)

In [ ]:
# Installation library

In [ ]:
# Importation des bibliothèques
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("datas/catnat_dirty.csv")

## Exercice 1 — Exploration et diagnostic

In [8]:
# 1. Afficher les dimensions du dataset (nombre de lignes et de colonnes).
df.shape

(17510, 18)

In [7]:
# 2. Affichez les types de données avec info()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17510 entries, 0 to 17509
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   DisNo.                   17510 non-null  object 
 1   Country                  17510 non-null  object 
 2   Region                   17161 non-null  object 
 3   Subregion                17510 non-null  object 
 4   Disaster Type            17510 non-null  object 
 5   Disaster Subtype         17510 non-null  object 
 6   Disaster Subgroup        17510 non-null  object 
 7   Event Name               3975 non-null   object 
 8   Start Year               17510 non-null  object 
 9   Start Month              16599 non-null  float64
 10  Total Deaths             12593 non-null  float64
 11  No. Injured              4507 non-null   float64
 12  Total Affected           12951 non-null  float64
 13  No. Homeless             2521 non-null   float64
 14  Total Damage ('000 US$

In [ ]:
# 3. Comptez les valeurs manquantes par colonne (nombre et pourcentage)
valeur_manquante = df.isnull().sum()
print("Nombre de valeurs manquantes par colonne:\n", valeur_manquante)

Nombre de valeurs manquantes par colonne:
 DisNo.                         0
Country                        0
Region                       349
Subregion                      0
Disaster Type                  0
Disaster Subtype               0
Disaster Subgroup              0
Event Name                 13535
Start Year                     0
Start Month                  911
Total Deaths                4917
No. Injured                13003
Total Affected              4559
No. Homeless               14989
Total Damage ('000 US$)    11880
Magnitude                  12261
Latitude                   14689
Longitude                  14689
dtype: int64


In [13]:
#4. Comptez le nombre de doublons
nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons dans le dataset : {nb_doublons}")

Nombre de doublons dans le dataset : 150


In [21]:
# 5. Affichez les valeurs uniques de Region — repérez les incohérences
unique_val_region = df["Region"].value_counts()
print("Valeurs uniques de la colonne 'Region':\n", unique_val_region)

Valeurs uniques de la colonne 'Region':
 Region
Asia          3754
Americas      2359
Africa        1688
Europe        1201
ASIA           792
asia           763
americas       504
AMERICAS       486
Asia           429
 Asia          396
Oceania        396
AFRICA         375
africa         370
 Asia          282
EUROPE         265
 Americas      248
europe         232
Americas       231
 Africa        202
Africa         181
 Americas      178
 Africa        140
 ASIA          116
Europe         105
 Europe        100
OCEANIA         84
 asia           82
ASIA            77
oceania         75
 Europe         75
asia            69
 asia           61
 ASIA           60
 AMERICAS       57
americas        56
AMERICAS        46
 americas       44
 AMERICAS       42
AFRICA          42
Oceania         41
 AFRICA         38
 Oceania        38
 africa         37
africa          37
 AFRICA         34
 americas       32
 europe         31
EUROPE          31
europe          30
 EUROPE         26
 a

In [22]:
unique_val_region2 = df["Region"].unique()
print("Valeurs uniques de la colonne 'Region':\n", unique_val_region2)

Valeurs uniques de la colonne 'Region':
 ['Asia ' 'Americas' 'Oceania' 'Asia' 'ASIA' ' Americas ' 'Africa'
 'Europe ' 'Europe' ' Americas' 'EUROPE' 'AMERICAS ' 'americas' 'AMERICAS'
 nan ' Asia ' ' Africa' 'Americas ' 'asia' ' Oceania' 'ASIA ' 'oceania'
 ' americas ' ' Asia' 'asia ' 'OCEANIA' 'africa' 'AFRICA' 'europe'
 'europe ' ' AMERICAS ' 'americas ' ' AMERICAS' ' Europe ' ' EUROPE'
 ' ASIA' ' Africa ' ' americas' 'africa ' ' asia' ' africa' ' AFRICA '
 'Africa ' ' AFRICA' 'Oceania ' ' asia ' 'AFRICA ' ' europe ' 'EUROPE '
 ' Europe' ' africa ' ' ASIA ' ' europe' ' Oceania ' 'OCEANIA '
 ' OCEANIA ' ' EUROPE ' 'oceania ' ' OCEANIA' ' oceania ' ' oceania']


**incohérences observées :** 
- `Region` : "ASIA", "asia", "Asia ", "EUROPE", "EUROPE ", etc. Majuscules, minuscules, espaces
- `Country` : "USA", "US", "United States", "United States of America", etc. variantes d'orthographe
- `oceania` : "Oceania", "oceania ", "OCEANIA", etc. doublons
- `nan`


In [19]:
# 6. Affichez les statistiques de Total Deaths — repérez les anomalies
stats_total_deaths = df["Total Deaths"].describe()
stats_total_deaths

count    1.259300e+04
mean     2.751663e+03
std      6.577989e+04
min     -3.600000e+04
25%      5.000000e+00
50%      1.800000e+01
75%      6.100000e+01
max      3.700000e+06
Name: Total Deaths, dtype: float64

### Questions
- Combien y a-t-il de doublons ?

> 150
- Quelles colonnes ont le plus de valeurs manquantes ? (Top 3)

> 1) No. Homeless = 14989, 
> 2) Latitude = 14689,
> 3) Longitude = 14689
- Combien de variantes différentes pour "Asia" ?
> 12 variantes différentes pour Asia.
> 'Asia '
'Asia'
'ASIA'
' Asia '
'asia'
'ASIA '
' Asia'
'asia '
' ASIA'
' asia'
' asia '
' ASIA '

- Y a-t-il des valeurs négatives dans Total Deaths ?
> oui il y a des valeurs négatives dans Total Deaths. (min = -3.600000e+04)


## Exercice 2 — Suppression des doublons

In [25]:
# 1. Affichez quelques lignes dupliquées pour vérifier

df[df.duplicated()]
df[df.duplicated(keep=False)].head(10)

,DisNo.,Country,Region,Subregion,Disaster Type,Disaster Subtype,Disaster Subgroup,Event Name,Start Year,Start Month,Total Deaths,No. Injured,Total Affected,No. Homeless,Total Damage ('000 US$),Magnitude,Latitude,Longitude
27,1969-0071-IND,India,ASIA,Southern Asia,Storm,Tropical cyclone,Meteorological,NaN,1969,5.0,600000.0,NaN,260000.0,NaN,8330.0,NaN,NaN,NaN
125,2005-0583-USA,USA,americas,Northern America,Flood,Riverine flood,Hydrological,NaN,2005,10.0,11.0,NaN,3000.0,NaN,NaN,38290.0,NaN,NaN
322,1963-0055-BEL,Belgium,Europe,Western Europe,Extreme Temperature,Cold wave,Meteorological,NaN,1963,NaN,12.0,NaN,NaN,NaN,NaN,-22.0,NaN,NaN
371,2023-0510-MNG,Mongolia,Asia,Eastern Asia,Flood,Flash flood,Hydrological,NaN,2023,8.0,4.0,NaN,1230.0,NaN,NaN,NaN,NaN,NaN
456,1996-0226-CHN,China,Asia,Eastern Asia,Storm,Tropical cyclone,Meteorological,Willie,1996,9.0,38.0,NaN,NaN,NaN,100000.0,130.0,NaN,NaN
489,2025-0173-THA,Thailand,NaN,South-eastern Asia,Storm,Severe weather,Meteorological,NaN,2025,3.0,NaN,NaN,4231.0,NaN,NaN,NaN,NaN,NaN
494,2018-0436-AGO,Angola,Africa,Sub-Saharan Africa,epidemic,Bacterial disease,Biological,Cholera,2018,10.0,2.0,NaN,139.0,NaN,NaN,NaN,NaN,NaN
570,2021-0596-PAK,Pakistan,Asia,Southern Asia,Flood,Flood (General),Hydrological,NaN,2021,9.0,19000.0,4.0,4.0,NaN,NaN,NaN,NaN,NaN
586,1989-0224-CHN,China,ASIA,Eastern Asia,Storm,Tropical cyclone,Meteorological,Brian,1989,10.0,31.0,700.0,700.0,NaN,NaN,NaN,NaN,NaN
617,2002-0406-JPN,JAPAN,Asia,Eastern Asia,Storm,Storm (General),Meteorological,NaN,2002,7.0,1.0,NaN,NaN,NaN,500.0,NaN,NaN,NaN
